# Replication Phase 1 - 4 Parquet Constrained Baseline

This notebook starts a rigorous but data-constrained replication of *Pairs Trading Using a Novel Graphical Matching Approach*.

Available data only:
- `prices.parquet`
- `pe_ratios.parquet`
- `risk_free.parquet`
- `universe.parquet`

Scope of this phase:
1. Build monthly candidate universes from historical constituents.
2. Select a value-filtered stock subset using P/E where available.
3. Score pairs with an Engle-Granger style residual stationarity proxy (ADF(0) t-stat approximation).
4. Build disjoint pairs via greedy max-weight matching (baseline approximation of graph matching).
5. Run monthly trading simulation with z-score entry/exit and produce diagnostics.

## Methodological notes and limitations

- This is a **baseline** and not yet a full faithful implementation of the paper.
- Matching step uses greedy disjoint matching, not exact Blossom maximum-weight matching.
- ADF p-values are not computed; we rank by the estimated ADF t-stat proxy on residuals.
- Returns are normalized spread-PnL style signals, so metrics should be interpreted as research diagnostics.
- The notebook is designed to be robust and extensible when additional data arrives.

In [ ]:
from __future__ import annotations

from pathlib import Path
import itertools
import math

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 180)

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'src').exists():
    ROOT = ROOT.parent

if not (ROOT / 'src').exists():
    raise RuntimeError('Could not locate project root containing src/.')

RAW_DIR = ROOT / 'src' / 'data' / 'raw'
OUT_DIR = ROOT / 'src' / 'data' / 'processed' / 'phase1_partial_replication'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {ROOT}')
print(f'Raw data dir: {RAW_DIR}')
print(f'Output dir  : {OUT_DIR}')

In [ ]:
prices = pd.read_parquet(RAW_DIR / 'prices.parquet')
pe_ratios = pd.read_parquet(RAW_DIR / 'pe_ratios.parquet')
risk_free = pd.read_parquet(RAW_DIR / 'risk_free.parquet')
universe = pd.read_parquet(RAW_DIR / 'universe.parquet')

prices.index = pd.to_datetime(prices.index)
pe_ratios.index = pd.to_datetime(pe_ratios.index)
risk_free.index = pd.to_datetime(risk_free.index)
universe['date'] = pd.to_datetime(universe['date'])

prices = prices.sort_index()
pe_ratios = pe_ratios.sort_index()
risk_free = risk_free.sort_index()
universe = universe.sort_values(['date', 'ticker'])

summary = pd.DataFrame({
    'dataset': ['prices', 'pe_ratios', 'risk_free', 'universe'],
    'rows': [len(prices), len(pe_ratios), len(risk_free), len(universe)],
    'cols': [prices.shape[1], pe_ratios.shape[1], risk_free.shape[1], universe.shape[1]],
    'start': [prices.index.min(), pe_ratios.index.min(), risk_free.index.min(), universe['date'].min()],
    'end': [prices.index.max(), pe_ratios.index.max(), risk_free.index.max(), universe['date'].max()],
})
summary

In [ ]:
# Research window and strategy parameters
BACKTEST_START = pd.Timestamp('2017-01-01')
BACKTEST_END = pd.Timestamp('2023-05-31')

LOOKBACK_DAYS = 504
MIN_OBS = 252
MIN_COVERAGE = 0.90
MAX_TICKERS = 120
MAX_PAIRS = 25

ADF_T_THRESHOLD = -2.50
ENTRY_Z = 2.0
EXIT_Z = 0.5
STOP_Z = 4.0

# Build safe aligned prices
prices = prices.replace([np.inf, -np.inf], np.nan)
prices = prices.where(prices > 0.0)
log_prices = np.log(prices)

# Rebalance dates come from historical membership snapshots
rebalance_dates = sorted(d for d in universe['date'].unique() if BACKTEST_START <= d <= BACKTEST_END)

print(f'Rebalance dates in window: {len(rebalance_dates)}')
print(f'First: {rebalance_dates[0] if rebalance_dates else None}')
print(f'Last : {rebalance_dates[-1] if rebalance_dates else None}')

In [ ]:
def fit_ols(y: np.ndarray, x: np.ndarray) -> tuple[float, float, np.ndarray]:
    """Return alpha, beta and residuals for y = alpha + beta*x."""
    x_mean = float(x.mean())
    y_mean = float(y.mean())
    denom = float(((x - x_mean) ** 2).sum())
    if denom <= 1e-12:
        return np.nan, np.nan, np.array([])
    beta = float(((x - x_mean) * (y - y_mean)).sum() / denom)
    alpha = y_mean - beta * x_mean
    resid = y - (alpha + beta * x)
    return alpha, beta, resid


def adf0_t_stat(series: np.ndarray) -> float:
    """Approximate ADF(0) t-stat using Delta e_t = c + phi * e_{t-1} + eps_t."""
    e = pd.Series(series).dropna().to_numpy(dtype=float)
    if len(e) < 30:
        return np.nan

    de = np.diff(e)
    lag = e[:-1]
    X = np.column_stack([np.ones_like(lag), lag])
    y = de

    try:
        beta_hat, *_ = np.linalg.lstsq(X, y, rcond=None)
        resid = y - X @ beta_hat
        dof = len(y) - X.shape[1]
        if dof <= 0:
            return np.nan

        s2 = float((resid @ resid) / dof)
        xtx_inv = np.linalg.inv(X.T @ X)
        var_beta = s2 * xtx_inv
        se_phi = float(np.sqrt(var_beta[1, 1]))
        if se_phi <= 1e-12:
            return np.nan

        t_phi = float(beta_hat[1] / se_phi)
        return t_phi
    except np.linalg.LinAlgError:
        return np.nan


def score_pair(s1: pd.Series, s2: pd.Series, min_obs: int = MIN_OBS) -> dict | None:
    """Evaluate both spread orientations and keep the one with lower ADF t-stat."""
    df = pd.concat([s1, s2], axis=1, keys=['a', 'b']).dropna()
    if len(df) < min_obs:
        return None

    y1 = df['a'].to_numpy(dtype=float)
    x1 = df['b'].to_numpy(dtype=float)
    alpha1, beta1, resid1 = fit_ols(y1, x1)
    t1 = adf0_t_stat(resid1)

    y2 = df['b'].to_numpy(dtype=float)
    x2 = df['a'].to_numpy(dtype=float)
    alpha2, beta2, resid2 = fit_ols(y2, x2)
    t2 = adf0_t_stat(resid2)

    candidates = []
    if not np.isnan(t1):
        candidates.append(('a_on_b', alpha1, beta1, resid1, t1))
    if not np.isnan(t2):
        candidates.append(('b_on_a', alpha2, beta2, resid2, t2))

    if not candidates:
        return None

    best = min(candidates, key=lambda x: x[4])
    orientation, alpha, beta, resid, t_stat = best

    sigma = float(np.std(resid, ddof=1))
    if not np.isfinite(sigma) or sigma <= 1e-8:
        return None

    return {
        'orientation': orientation,
        'alpha': float(alpha),
        'beta': float(beta),
        'mu': float(np.mean(resid)),
        'sigma': sigma,
        'adf_t_stat': float(t_stat),
        'n_obs': int(len(df)),
        'weight': float(-t_stat),
    }

In [ ]:
def select_candidates(
    hist_log_prices: pd.DataFrame,
    pe_ratios: pd.DataFrame,
    formation_end: pd.Timestamp,
    min_coverage: float = MIN_COVERAGE,
    max_tickers: int = MAX_TICKERS,
) -> list[str]:
    coverage = hist_log_prices.notna().mean()
    eligible = coverage[coverage >= min_coverage].index.tolist()
    if not eligible:
        return []

    pe_eligible = [t for t in eligible if t in pe_ratios.columns]
    if not pe_eligible:
        return coverage[coverage >= min_coverage].sort_values(ascending=False).index.tolist()[:max_tickers]

    pe_slice = pe_ratios.reindex(hist_log_prices.index)[pe_eligible]
    pe_hist = pe_slice[pe_slice.index <= formation_end]
    if pe_hist.empty:
        pe_snapshot = pd.Series(dtype=float)
    else:
        pe_snapshot = pe_hist.ffill().iloc[-1]
    pe_snapshot = pe_snapshot[np.isfinite(pe_snapshot)]
    pe_snapshot = pe_snapshot[pe_snapshot > 0]

    # Trim extreme P/E values to avoid pathological outliers in this constrained setup.
    if len(pe_snapshot) >= 20:
        upper = pe_snapshot.quantile(0.95)
        pe_snapshot = pe_snapshot[pe_snapshot <= upper]

    value_ranked = pe_snapshot.sort_values().index.tolist()

    if len(value_ranked) >= max_tickers:
        return value_ranked[:max_tickers]

    # Fallback: fill remaining slots with best coverage names not already selected.
    remaining = [t for t in coverage.sort_values(ascending=False).index if t in eligible and t not in value_ranked]
    return (value_ranked + remaining)[:max_tickers]


def build_edges_for_date(hist_log_prices: pd.DataFrame) -> pd.DataFrame:
    tickers = hist_log_prices.columns.tolist()
    records: list[dict] = []

    for i, j in itertools.combinations(range(len(tickers)), 2):
        t_i = tickers[i]
        t_j = tickers[j]
        scored = score_pair(hist_log_prices[t_i], hist_log_prices[t_j], min_obs=MIN_OBS)
        if scored is None:
            continue
        if scored['adf_t_stat'] >= ADF_T_THRESHOLD:
            continue

        rec = {
            'asset_a': t_i,
            'asset_b': t_j,
            **scored,
        }
        records.append(rec)

    if not records:
        return pd.DataFrame(columns=['asset_a', 'asset_b', 'orientation', 'alpha', 'beta', 'mu', 'sigma', 'adf_t_stat', 'n_obs', 'weight'])

    edges = pd.DataFrame(records).sort_values('weight', ascending=False).reset_index(drop=True)
    return edges


def greedy_disjoint_matching(edges: pd.DataFrame, max_pairs: int = MAX_PAIRS) -> pd.DataFrame:
    used = set()
    chosen_rows = []

    for _, row in edges.iterrows():
        a = row['asset_a']
        b = row['asset_b']
        if a in used or b in used:
            continue
        chosen_rows.append(row)
        used.add(a)
        used.add(b)
        if len(chosen_rows) >= max_pairs:
            break

    if not chosen_rows:
        return pd.DataFrame(columns=edges.columns)

    return pd.DataFrame(chosen_rows).reset_index(drop=True)

In [ ]:
def build_positions(z: pd.Series, entry: float = ENTRY_Z, exit_: float = EXIT_Z, stop: float = STOP_Z) -> pd.Series:
    pos = np.zeros(len(z), dtype=float)
    current = 0.0

    for i, val in enumerate(z.to_numpy(dtype=float)):
        if not np.isfinite(val):
            current = 0.0
        elif current == 0.0:
            if val >= entry:
                current = -1.0
            elif val <= -entry:
                current = 1.0
        else:
            if abs(val) <= exit_ or abs(val) >= stop:
                current = 0.0

        pos[i] = current

    return pd.Series(pos, index=z.index)


def simulate_month(
    selected_pairs: pd.DataFrame,
    log_price_panel: pd.DataFrame,
    trade_index: pd.DatetimeIndex,
) -> pd.Series:
    if selected_pairs.empty or len(trade_index) == 0:
        return pd.Series(dtype=float)

    pair_returns = []

    for _, pair in selected_pairs.iterrows():
        a = pair['asset_a']
        b = pair['asset_b']

        px = log_price_panel[[a, b]].reindex(trade_index).dropna()
        if len(px) < 2:
            continue

        if pair['orientation'] == 'a_on_b':
            y = px[a]
            x = px[b]
        else:
            y = px[b]
            x = px[a]

        spread = y - (pair['alpha'] + pair['beta'] * x)
        z = (spread - pair['mu']) / pair['sigma']

        position = build_positions(z)
        pnl = position.shift(1).fillna(0.0) * spread.diff().fillna(0.0)

        # Normalize by formation spread sigma and clip to reduce instability in this phase.
        ret = (pnl / pair['sigma']).clip(-0.05, 0.05)
        pair_returns.append(ret.reindex(trade_index).fillna(0.0))

    if not pair_returns:
        return pd.Series(dtype=float)

    panel = pd.concat(pair_returns, axis=1)
    port = panel.mean(axis=1)
    port.name = 'strategy_ret'
    return port


def infer_daily_risk_free(rf: pd.Series) -> pd.Series:
    s = rf.dropna().astype(float).copy()
    if s.empty:
        return s

    med = float(s.median())

    # Heuristic conversion:
    # - if values look like percent annual yields (e.g. 2.5), divide by 100 then by 252
    # - if values look like decimal annual yields (e.g. 0.03), divide by 252
    # - otherwise assume already daily decimal returns
    if med > 0.20:
        return (s / 100.0) / 252.0
    if med > 0.02:
        return s / 252.0
    return s


def annualized_return(r: pd.Series) -> float:
    r = r.dropna()
    if len(r) == 0:
        return np.nan
    return float((1.0 + r).prod() ** (252.0 / len(r)) - 1.0)


def annualized_vol(r: pd.Series) -> float:
    r = r.dropna()
    if len(r) == 0:
        return np.nan
    return float(r.std(ddof=0) * math.sqrt(252.0))


def sharpe_ratio(r: pd.Series, rf_daily: pd.Series | None = None) -> float:
    r = r.dropna()
    if len(r) == 0:
        return np.nan

    if rf_daily is None or rf_daily.empty:
        ex = r
    else:
        ex = r.sub(rf_daily.reindex(r.index).fillna(0.0), fill_value=0.0)

    vol = annualized_vol(ex)
    if not np.isfinite(vol) or vol <= 1e-12:
        return np.nan

    return annualized_return(ex) / vol

In [ ]:
all_portfolio_returns: list[pd.Series] = []
all_edges: list[pd.DataFrame] = []
all_selected_pairs: list[pd.DataFrame] = []
diagnostics: list[dict] = []
pair_sets_by_rebal: dict[pd.Timestamp, set[tuple[str, str]]] = {}

for reb_date, next_reb_date in zip(rebalance_dates[:-1], rebalance_dates[1:]):
    formation_candidates = prices.index[prices.index <= reb_date]
    trade_end_candidates = prices.index[prices.index <= next_reb_date]

    if len(formation_candidates) == 0 or len(trade_end_candidates) == 0:
        continue

    formation_end = formation_candidates.max()
    trade_end = trade_end_candidates.max()

    hist = log_prices[log_prices.index <= formation_end].tail(LOOKBACK_DAYS)
    active = set(universe.loc[universe['date'] == reb_date, 'ticker'])

    cols = [c for c in hist.columns if c in active]
    if len(cols) < 20:
        diagnostics.append({
            'rebalance_date': reb_date,
            'formation_end': formation_end,
            'n_active_universe': len(active),
            'n_hist_assets': len(cols),
            'n_candidates': 0,
            'n_edges': 0,
            'n_pairs': 0,
            'trade_days': 0,
            'status': 'skip_too_few_assets',
        })
        continue

    hist_active = hist[cols]
    candidates = select_candidates(hist_active, pe_ratios, formation_end)
    if len(candidates) < 10:
        diagnostics.append({
            'rebalance_date': reb_date,
            'formation_end': formation_end,
            'n_active_universe': len(active),
            'n_hist_assets': len(cols),
            'n_candidates': len(candidates),
            'n_edges': 0,
            'n_pairs': 0,
            'trade_days': 0,
            'status': 'skip_too_few_candidates',
        })
        continue

    edges = build_edges_for_date(hist_active[candidates])
    selected = greedy_disjoint_matching(edges, max_pairs=MAX_PAIRS)

    trade_index = prices.index[(prices.index > formation_end) & (prices.index <= trade_end)]
    monthly_ret = simulate_month(selected, log_prices, trade_index)

    if not edges.empty:
        tmp_edges = edges.copy()
        tmp_edges['rebalance_date'] = reb_date
        all_edges.append(tmp_edges)

    if not selected.empty:
        tmp_sel = selected.copy()
        tmp_sel['rebalance_date'] = reb_date
        all_selected_pairs.append(tmp_sel)

        pair_set = {tuple(sorted((r['asset_a'], r['asset_b']))) for _, r in selected.iterrows()}
        pair_sets_by_rebal[reb_date] = pair_set

    if not monthly_ret.empty:
        all_portfolio_returns.append(monthly_ret)

    diagnostics.append({
        'rebalance_date': reb_date,
        'formation_end': formation_end,
        'n_active_universe': len(active),
        'n_hist_assets': len(cols),
        'n_candidates': len(candidates),
        'n_edges': len(edges),
        'n_pairs': len(selected),
        'trade_days': len(trade_index),
        'status': 'ok' if len(selected) > 0 and len(monthly_ret) > 0 else 'ok_no_trades',
    })

strategy_returns = pd.concat(all_portfolio_returns).sort_index() if all_portfolio_returns else pd.Series(dtype=float)
strategy_returns = strategy_returns[~strategy_returns.index.duplicated(keep='first')]

diag_df = pd.DataFrame(diagnostics).sort_values('rebalance_date').reset_index(drop=True)
edges_df = pd.concat(all_edges, ignore_index=True) if all_edges else pd.DataFrame()
selected_df = pd.concat(all_selected_pairs, ignore_index=True) if all_selected_pairs else pd.DataFrame()

print(f'Rebalances processed: {len(diag_df)}')
print(f'Rebalances with selected pairs: {(diag_df["n_pairs"] > 0).sum() if not diag_df.empty else 0}')
print(f'Strategy daily rows: {len(strategy_returns)}')
diag_df.tail(10)

In [ ]:
rf_series = risk_free.iloc[:, 0] if risk_free.shape[1] > 0 else pd.Series(dtype=float)
rf_daily = infer_daily_risk_free(rf_series)

# Benchmark proxy from broad universe center as placeholder when benchmark parquet is unavailable here.
# We use equal-weight return of currently active universe as a comparison baseline.
benchmark_proxy = prices.pct_change(fill_method=None).mean(axis=1).reindex(strategy_returns.index).fillna(0.0)

if not strategy_returns.empty:
    report = pd.DataFrame({
        'metric': [
            'n_days',
            'ann_return',
            'ann_vol',
            'sharpe_vs_rf',
            'max_drawdown',
            'avg_daily_ret',
            'avg_abs_daily_ret',
        ],
        'strategy': [
            len(strategy_returns),
            annualized_return(strategy_returns),
            annualized_vol(strategy_returns),
            sharpe_ratio(strategy_returns, rf_daily),
            ((1.0 + strategy_returns).cumprod() / (1.0 + strategy_returns).cumprod().cummax() - 1.0).min(),
            strategy_returns.mean(),
            strategy_returns.abs().mean(),
        ],
        'benchmark_proxy': [
            len(benchmark_proxy),
            annualized_return(benchmark_proxy),
            annualized_vol(benchmark_proxy),
            sharpe_ratio(benchmark_proxy, rf_daily),
            ((1.0 + benchmark_proxy).cumprod() / (1.0 + benchmark_proxy).cumprod().cummax() - 1.0).min(),
            benchmark_proxy.mean(),
            benchmark_proxy.abs().mean(),
        ]
    })

    equity = pd.DataFrame({
        'strategy_equity': (1.0 + strategy_returns).cumprod(),
        'benchmark_proxy_equity': (1.0 + benchmark_proxy).cumprod(),
    })

    display(report)
    display(equity.tail())
else:
    print('No strategy returns produced. Check diagnostics and thresholds.')

In [ ]:
def pair_set_retention(pair_sets: dict[pd.Timestamp, set[tuple[str, str]]]) -> pd.DataFrame:
    dates = sorted(pair_sets.keys())
    rows = []
    for d0, d1 in zip(dates[:-1], dates[1:]):
        s0 = pair_sets[d0]
        s1 = pair_sets[d1]
        inter = len(s0.intersection(s1))
        union = len(s0.union(s1)) if s0 or s1 else 0
        jacc = inter / union if union > 0 else np.nan
        rows.append({
            'from_rebalance': d0,
            'to_rebalance': d1,
            'pairs_prev': len(s0),
            'pairs_next': len(s1),
            'pairs_overlap': inter,
            'jaccard_retention': jacc,
        })
    return pd.DataFrame(rows)

retention_df = pair_set_retention(pair_sets_by_rebal)
retention_df.head(10)

In [ ]:
# Persist all research outputs for reproducibility
diag_df.to_csv(OUT_DIR / 'diagnostics.csv', index=False)

if not edges_df.empty:
    edges_df.to_parquet(OUT_DIR / 'candidate_edges.parquet', index=False)

if not selected_df.empty:
    selected_df.to_parquet(OUT_DIR / 'selected_pairs.parquet', index=False)

if not strategy_returns.empty:
    strategy_returns.to_frame('strategy_ret').to_parquet(OUT_DIR / 'strategy_returns.parquet')

if not retention_df.empty:
    retention_df.to_csv(OUT_DIR / 'pair_retention.csv', index=False)

print('Saved outputs to:')
for p in sorted(OUT_DIR.glob('*')):
    print(' -', p.name)

## What to do next

1. Replace greedy matching with exact maximum-weight matching (graph algorithm) to match the paper.
2. Add benchmark total-return series and static metadata for sector-neutral constraints.
3. Replace ADF t-stat proxy with exact ADF test (with lag selection and p-values).
4. Add transaction costs, borrow constraints, and realistic capital allocation.
5. Run sensitivity grids on thresholds (entry/exit/stop, ADF cutoff, max pairs).